In [1]:
import polars as pl
import polars.selectors as cs
from scipy.stats import norm
from plotly.offline import init_notebook_mode

init_notebook_mode(connected=True)
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import calendar

In [ ]:
cpiu = (
    pl.read_excel("cpiu.xlsx", read_options={"header_row": 11})
    .rename({m: str(i) for i, m in enumerate(calendar.month_abbr[1:])})
    .unpivot(
        map(str, range(12)),
        index=["Year", "Annual"],
        variable_name="month",
        value_name="cpiu",
    )
    .sort("Year")
    .with_columns(
        pl.col("month").str.to_integer(),
        annual_computed=pl.col("cpiu").mean().over("Year"),
    )
    .with_columns(
        annual_delta=pl.when(
            pl.col("annual_computed").shift(-1) != pl.col("annual_computed")
        )
        .then(
            (pl.col("annual_computed").shift(-1) - pl.col("annual_computed"))
            / pl.col("annual_computed")
        )
        .backward_fill()
        .forward_fill()
    )
    .with_columns(
        filled_cpiu=pl.coalesce(
            pl.col("cpiu"),
            (1 + pl.col("annual_delta") / 12).pow(pl.col("month"))
            * pl.col("cpiu").first().over("Year"),
        )
    )
    .with_columns(
        # We want month to be 0-based when doing the exponentiation above but need it to be 1-based for joins
        month=pl.col("month") + 1,
        eoy_delta=(pl.col("filled_cpiu").last() - pl.col("filled_cpiu"))
        / pl.col("filled_cpiu"),
    )
)

In [3]:
budget = (
    pl.concat(
        [
            pl.read_json("budget2024.json", infer_schema_length=None),
            pl.read_json("budget2025.json", infer_schema_length=None),
        ]
    )
    .unnest("data")
    .unnest("budgetVsActualV2")
)

In [4]:
def extract_cashflow(cashflow: pl.Series):
    return (
        cashflow.explode()
        .struct.unnest()
        .select("subAccounts", headline_category="category")
        .explode("subAccounts")
        .unnest("subAccounts")
        .select(
            "headline_category",
            "category",
            pl.col("annual").struct.field("transactions"),
        )
        .explode("transactions")
        .unnest("transactions")
        .with_columns(pl.col("date").str.to_date())
        .drop(["transactionType", "__typename"])
        .drop_nulls()
    )


expenses = extract_cashflow(budget["expenses"])


def aggregate_cashflow(cashflow: pl.DataFrame) -> pl.DataFrame:
    return cashflow.group_by(
        year=pl.col("date").dt.year(), month=pl.col("date").dt.month()
    ).agg(pl.col("amount").sum())


# Simulate next year.
MONTHS_TO_SIMULATE = 12
SIMULATIONS = 10000
UNPAID_BILLS = 8828.77
# Based on the "Operation" account pulled from Daisy dashboard on 2025-11-18
STARTING_BALANCE = 22873.94 - UNPAID_BILLS
# Simulate budget increases between 1% - 5% in increments of 1 percentage point.
BUDGET_INCREASES: pl.Expr = (
    pl.int_ranges(-5, 7)
    .alias("budget_increase")
    .list.eval((pl.element() / 100).round(2))
)
cashflow_by_month = (
    aggregate_cashflow(extract_cashflow(budget["incomes"]))
    .with_columns(BUDGET_INCREASES)
    .explode("budget_increase")
    .with_columns(pl.col("amount") * (1 + pl.col("budget_increase")))
    .join(
        aggregate_cashflow(expenses)
        .join(cpiu, left_on=["year", "month"], right_on=["Year", "month"])
        .select("year", "month", pl.col("amount") * (1 + pl.col("eoy_delta"))),
        on=["year", "month"],
        how="full",
        suffix="_expense",
    )
    .select(
        "budget_increase",
        "year",
        "month",
        pl.col("amount").fill_null(0) - pl.col("amount_expense").fill_null(0),
    )
)
simulations = (
    cashflow_by_month.group_by("budget_increase")
    .agg(pl.col("amount"))
    .with_columns(simulation_id=pl.int_ranges(0, SIMULATIONS))
    .explode("simulation_id")
    .with_columns(pl.col("amount").list.sample(MONTHS_TO_SIMULATE, shuffle=True))
    .explode("amount")
    .with_columns(
        pl.row_index("month").over("budget_increase", "simulation_id"),
        pl.col("amount").cum_sum().over("budget_increase", "simulation_id")
        + STARTING_BALANCE,
    )
)

In [ ]:
monthly_cashflow_by_budget_increase = (
    cashflow_by_month.sort("month")
    .pivot("month", index=["budget_increase", "year"], values="amount")
    .drop("year")
)
from functools import reduce


monthly_simulations = (
    reduce(
        lambda df, c: df.join(
            monthly_cashflow_by_budget_increase.select("budget_increase", c),
            on="budget_increase",
        ),
        monthly_cashflow_by_budget_increase.select(cs.digit()).columns,
        monthly_cashflow_by_budget_increase.select("budget_increase"),
    )
    .with_columns(pl.row_index("simulation_id").over("budget_increase"))
    .unpivot(
        cs.digit(),
        index=["budget_increase", "simulation_id"],
        variable_name="month",
        value_name="amount",
    )
    .sort("budget_increase", "simulation_id", pl.col("month").str.to_integer())
    .with_columns(
        pl.col("amount").cum_sum().over("budget_increase", "simulation_id")
        + STARTING_BALANCE
    )
)

In [6]:
simulations = monthly_simulations

In [ ]:
simulations_by_month = (
    simulations.group_by("month", "budget_increase")
    .agg(
        amount_average=pl.col("amount").mean(),
        amount_min=pl.col("amount").min(),
        amount_max=pl.col("amount").max(),
    )
    .sort("budget_increase", "month")
)

TRENDLINE_FIG_COL_COUNT = 2

unique_budget_increases = simulations["budget_increase"].unique()

fig = make_subplots(
    rows=int(len(unique_budget_increases) / TRENDLINE_FIG_COL_COUNT),
    cols=TRENDLINE_FIG_COL_COUNT,
    shared_yaxes="all",
    subplot_titles=[
        f"budget_increase={budget_increase}"
        for budget_increase in unique_budget_increases
    ],
)
for i, budget_increase in (
    unique_budget_increases.to_frame().with_row_index().iter_rows()
):
    sims = simulations_by_month.filter(pl.col("budget_increase") == budget_increase)
    row = int(i / TRENDLINE_FIG_COL_COUNT) + 1
    col = i % TRENDLINE_FIG_COL_COUNT + 1
    fig.add_trace(
        go.Scatter(
            x=pl.concat([sims["month"], sims["month"].reverse()]),
            y=pl.concat([sims["amount_min"], sims["amount_max"].reverse()]),
            name=f"min/max {budget_increase}",
            fill="toself",
        ),
        row=row,
        col=col,
    )
    fig.add_trace(
        go.Scatter(x=sims["month"], y=sims["amount_average"], name=budget_increase),
        row=row,
        col=col,
    )
    fig.update_xaxes(title_text="month", row=row, col=col)
fig.update_layout(height=1000, legend=go.layout.Legend(title="budget_increase"))
fig.show()

In [8]:
simulation_mins = simulations.group_by("budget_increase", "simulation_id").agg(
    pl.col("amount").min()
)

In [9]:
fig = px.pie(
    simulation_mins.group_by(
        "budget_increase",
        ruinous=pl.col("amount") < 0,
    )
    .len("simulation_count")
    .with_columns(
        ruinous=pl.when("ruinous")
        .then(pl.lit("Special Assessment"))
        .otherwise(pl.lit("Safe"))
    )
    .sort("budget_increase"),
    names="ruinous",
    values="simulation_count",
    facet_col="budget_increase",
    facet_col_wrap=3,
    title="Likelihood of Special Assessment",
    color_discrete_sequence=["#4B08AF", "#32965D"],
    height=1000,
)
fig.show(renderer="notebook_connected")

# Special Assessment
95% confidence that the special assessment--if there is one--will be less than `amount`

In [ ]:
# Inflation estimation from https://www.federalreserve.gov/monetarypolicy/files/fomcprojtabl20250917.pdf
INFLATION = 0.026
assessment_recommendations = (
    simulation_mins.filter(pl.col("amount") < 0)
    .sort("budget_increase")
    .with_columns(-pl.col("amount"))
    .group_by("budget_increase")
    .agg(
        amount_95_conf=pl.col("amount").mean()
        + pl.col("amount").std() * norm.ppf(0.95),
        amount_99_conf=pl.col("amount").mean()
        + pl.col("amount").std() * norm.ppf(0.99),
    )
    .with_columns(
        amount_95_conf_with_inflation=pl.col("amount_95_conf") * (1 + INFLATION),
        amount_99_conf_with_inflation=pl.col("amount_99_conf") * (1 + INFLATION),
    )
)
assessment_recommendations.style

budget_increase,amount_95_conf,amount_99_conf,amount_95_conf_with_inflation,amount_99_conf_with_inflation
-0.05,23851.710109994718,29397.019864437374,24471.85457285458,30161.342380912745
-0.04,21949.82867190026,27193.440612579525,22520.524217369668,27900.470068506595
-0.03,20427.301464459975,25344.891011503223,20958.411302535937,26003.858177802307
-0.02,19345.356010164527,23815.88029737592,19848.335266428807,24435.093185107693
-0.01,17648.370059291698,21851.863136299773,18107.227680833283,22420.01157784357
0.0,16165.467693342654,20067.618259316652,16585.769853369562,20589.376334058885
0.01,14567.652977425329,18226.51026845641,14946.411954838388,18700.399535436278
0.02,14159.292171431098,17453.961800259287,14527.433767888308,17907.76480706603
0.03,12657.012129072275,15704.267214321793,12986.094444428154,16112.57816189416
0.04,11392.89165612475,14155.963703694219,11689.106839183993,14524.01875999027


In [11]:
fig = px.bar(
    expenses.with_columns(
        chart_date=pl.date(pl.col("date").dt.year(), pl.col("date").dt.month(), 1)
    ),
    x="chart_date",
    y="amount",
    color="headline_category",
)
fig.show(renderer="notebook_connected")

In [12]:
fig = px.bar(
    expenses.with_columns(
        chart_date=pl.date(pl.col("date").dt.year(), pl.col("date").dt.month(), 1)
    ),
    x="chart_date",
    y="amount",
    color="category",
)
fig.show(renderer="notebook_connected")